In [1]:
require "fileutils"

false

In [3]:
CHAPTER_DIR = 'images'
PAGE_DIR = 'pages'
MANGA = false

if Dir.exist?(PAGE_DIR)
  FileUtils.rm_rf(PAGE_DIR)
end

FileUtils.mkdir_p(PAGE_DIR)

chapters = Dir.glob("#{CHAPTER_DIR}/*").filter { |f| File.directory?(f) }.sort
chapters.each_with_index do |chapter, chap_idx|
  chapter_name = File.basename(chapter)
  page_output_dir = "#{PAGE_DIR}/#{chapter_name}"
  FileUtils.mkdir_p(page_output_dir)
  pages = Dir.glob("#{chapter}/*").filter { |f| File.file?(f) }.sort
  pages.each_with_index do |page, idx|
    prev_page = idx > 0 ? File.basename(pages[idx - 1]).split('.').first : nil
    prev_chapter = nil
    if prev_page.nil?
      if chap_idx > 0
        prev_chapter = File.basename(chapters[chap_idx - 1])
        prev_pages = Dir.glob("#{CHAPTER_DIR}/#{prev_chapter}/*").filter { |f| File.file?(f) }.sort
        prev_page = File.basename(prev_pages.last).split('.').first
      end
    end
    next_page = idx < pages.length - 1 ? File.basename(pages[idx + 1]).split('.').first : nil
    next_chapter = nil
    if next_page.nil?
      if chap_idx < chapters.length - 1
        next_chapter = File.basename(chapters[chap_idx + 1])
        next_pages = Dir.glob("#{CHAPTER_DIR}/#{next_chapter}/*").filter { |f| File.file?(f) }.sort
        next_page = File.basename(next_pages.first).split('.').first
      end
    end
    next_link = nil
    if next_page
      next_link = "#{next_chapter ? '../' + next_chapter + '/' : ''}#{next_page}.html"
    end
    prev_link = nil
    if prev_page
      prev_link = "#{prev_chapter ? '../' + prev_chapter + '/' : ''}#{prev_page}.html"
    end
    page_name = File.basename(page)
    output_file = "#{page_output_dir}/#{page_name.split('.').first}.html"
    html = <<~HTML
      <!DOCTYPE html>
      <html lang="en">
        <head>
          <meta charset="UTF-8">
          <meta name="viewport" content="width=device-width, initial-scale=1.0">
          <title>#{chapter_name} - #{page_name}</title>
        </head>
        <body style="text-align: center; background-color: #000; color: #fff;">
          #{next_page ? "<a href='#{next_link}'>" : ''}
          <img src="../../#{CHAPTER_DIR}/#{chapter_name}/#{page_name}" style="max-height: 42.5rem; height: auto; max-width: 100%;" />
          #{next_page ? "</a>" : ''}
          <nav style="margin: 0">
            #{
              MANGA ? (
                (next_page ? "<a href='#{next_link}' class='next' style='color: #fff;'>&laquo; Next</a>" : '') \
                + "<a href='../../index.html' style='color: #fff; margin: 0 10px;'>Index</a>" \
                + (prev_page ? "<a href='#{prev_link}' class='prev' style='color: #fff;'>Prev &raquo;</a>" : '')
              ) : (
                (prev_page ? "<a href='#{prev_link}' class='prev' style='color: #fff;'>&laquo; Prev</a>" : '') \
                + "<a href='../../index.html' style='color: #fff; margin: 0 10px;'>Index</a>" \
                + (next_page ? "<a href='#{next_link}' class='next' style='color: #fff;'>Next &raquo;</a>" : '')
              )
            }
          </nav>
          <script src="../../js/nav.js"></script>
          #{MANGA ? '<p style="margin: 0;">("Next" and "Prev" are swapped to match the Manga reading direction.)</p>' : ''}
          <progress value='#{idx + 1}' max='#{pages.length}' style='width: 80%; direction: #{MANGA ? "rtl" : "ltr"};' aria-label="Page progress"></progress>
          <p>#{idx + 1}/#{pages.length}</p>
        </body>
      </html>
    HTML
    File.write(output_file, html)
  end
end

(irb): warning: already initialized constant Object::CHAPTER_DIR
(irb):1: warning: already initialized constant Object::PAGE_DIR
(irb):1: warning: previous definition of PAGE_DIR was here
(irb):2: warning: already initialized constant Object::MANGA
(irb):2: warning: previous definition of MANGA was here


["images/v001", "images/v002", "images/v003", "images/v004", "images/v005", "images/v006", "images/v007"]

In [ ]:
readings = [
  {
    week: 1,
    chapters: [1, 2, 3],
    type: "Volume"
  },
  {
    week: 2,
    chapters: [4, 5],
    type: "Volume"
  },
  {
    week: 3,
    chapters: [6, 7],
    type: "Volume"
  }
]

maps = [
  {
    chapter: 3,
    pages: [
      197
    ]
  },
  {
    chapter: 4,
    pages: [
      217,
      218,
    ]
  }
]
index_html = <<~HTML
  <!DOCTYPE html>
  <html lang="en">
    <head>
      <meta charset="UTF-8">
      <meta name="viewport" content="width=device-width, initial-scale=1.0">
      <title>Index</title>
    </head>
    <body style="text-align: center; background-color: #000; color: #fff;">
      <h1>Index</h1>
HTML

readings.each do |reading|
  index_html += <<~HTML
    <h2>Week #{reading[:week]}</h2>
    #{
      reading[:chapters].map { |chap| "<a href='pages/#{reading[:type][0].downcase}#{'%03d' % chap}/001.html' style='color: #fff;'>Volume #{chap}</a>" }.join("<br>")
    }
  HTML
end

if maps.any?
  index_html += <<~HTML
    <h2>Maps</h2>
    <a href='maps.html' style='color: #fff;'>Maps</a>
  HTML
end

index_html += <<~HTML
    </body>
  </html>
HTML

File.write("index.html", index_html)

715

In [5]:
maps_html = <<~HTML
  <!DOCTYPE html>
  <html lang="en">
    <head>
      <meta charset="UTF-8">
      <meta name="viewport" content="width=device-width, initial-scale=1.0">
      <title>Maps</title>
    </head>
    <body style="text-align: center; background-color: #000; color: #fff;">
      <h1>Maps</h1>
      #{
        maps.map do |map|
          <<~HTM
            <h2>#{readings[0][:type]} #{map[:chapter]}</h2>
            #{
              map[:pages].map { |page| "<img src='#{CHAPTER_DIR}/#{readings[0][:type][0].downcase}#{'%03d' % map[:chapter]}/#{'%03d' % page}.jpg' style='max-width: 80%; height: auto; margin-bottom: 20px;' />" }.join
            }
          HTM
        end.join
      }
      <nav style="margin: 0">
        <a href='index.html' style='color: #fff; margin: 0 10px;'>Index</a>
      </nav>
    </body>
  </html>
HTML

File.write("maps.html", maps_html)

720